# Library service trajectories

Open this notebook in Google Colab and choose **Runtime → Run all**. When prompted, upload `colab-reproduction.zip` from the repository release.

When the run finishes, Colab downloads `reproduction-output.zip` with the reproduced tables, figure data, Figures 1–4 and a short run summary.

In [ ]:
# @title
from pathlib import Path
import sys, subprocess, zipfile

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'checksums.sha256').is_file()), None)
if root is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Open this notebook from the repository folder.')
    print('Upload colab-reproduction.zip.')
    uploads = files.upload()
    archives = [name for name in uploads if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload one ZIP file.')
    destination = Path.cwd() / 'reproduction'
    with zipfile.ZipFile(archives[0]) as archive:
        members = archive.infolist()
        top_levels = {
            Path(item.filename).parts[0]
            for item in members
            if Path(item.filename).parts
        }
        if top_levels != {'library-service-trajectories'}:
            raise ValueError('The reproduction bundle has an unexpected folder structure.')
        for item in members:
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError('The reproduction bundle contains an invalid path.')
            archive.extract(item, destination)
    root = destination / 'library-service-trajectories'

assert (root / 'checksums.sha256').is_file()
for folder in ('results', 'figure-data', 'figures'):
    (root / folder).mkdir(parents=True, exist_ok=True)

sys.dont_write_bytecode = True
sys.path.insert(0, str(root / 'code'))


## Check inputs

The notebook verifies the eight reference inputs before running the analysis.

In [ ]:
# @title
import importlib.util
if any(importlib.util.find_spec(name) is None for name in ['numpy', 'matplotlib']):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements.txt')])
from check import check_inputs, check_results
input_summary = check_inputs(root)
input_summary

## Reproduce results

The analysis recreates the reported comparisons and output tables.

In [ ]:
from reproduce import reproduce
parity = reproduce(root)
result_summary = check_results(root)
result_summary

## Figures 1–4

Figures 1–4 are regenerated from the reproduced results.

In [ ]:
subprocess.check_call([sys.executable, '-B', str(root / 'code/figures.py')])
from IPython.display import SVG, display
for number in range(1, 5):
    display(SVG(filename=str(root / 'figures' / f'figure-{number}.svg')))

## Main results

The primary population-weighted difference is [9.97, 10.16] percentage points; the equal-territory difference is [1.58, 3.58]. For the later baseline, the population-weighted difference is [−2.98, −2.80].

Field definitions and source notes are available in the [data guide](../data/README.md) and [source terms](../licenses/third-party.md).

In [ ]:
# @title Download reproduced outputs
import hashlib
import json
import platform
import zipfile
from datetime import datetime, timezone

import matplotlib
import numpy as np

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

output_members = []
for folder in ("results", "figure-data", "figures"):
    for path in sorted((root / folder).iterdir()):
        if path.is_file():
            output_members.append(path)

bundle_path = None
if "archives" in globals() and len(archives) == 1:
    candidate = Path(archives[0])
    if candidate.is_file():
        bundle_path = candidate

summary = {
    "completed": True,
    "repository": "evidenceworks/library-service-trajectories",
    "input_bundle": {
        "filename": bundle_path.name if bundle_path else "colab-reproduction.zip",
        "sha256": _sha256(bundle_path) if bundle_path else None,
    },
    "checks": {
        "inputs": input_summary,
        "verified_result_objects": len(parity),
        "results": result_summary,
        "figures_regenerated": 4,
    },
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
        "platform": platform.platform(),
    },
    "outputs": [
        {
            "path": path.relative_to(root).as_posix(),
            "sha256": _sha256(path),
            "bytes": path.stat().st_size,
        }
        for path in output_members
    ],
    "completed_utc": datetime.now(timezone.utc).isoformat(),
}

summary_path = Path.cwd() / "reproduction-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

output_zip = Path.cwd() / "reproduction-output.zip"
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(summary_path, "reproduction-summary.json")
    for path in output_members:
        archive.write(path, path.relative_to(root).as_posix())

print("\nReproduction completed successfully.")
print(f"Checked {len(parity)} result objects and regenerated Figures 1–4.")
print(f"Created {output_zip.name} with {1 + len(output_members)} files.")

try:
    from google.colab import files
    files.download(str(output_zip))
except ImportError:
    print(f"Output saved to: {output_zip}")
